## 04 - Business Analysis and Key Metrics


In [ ]:
print("Delivered sales shape:", delivered_sales.shape)
print("Number of unique orders:", delivered_sales["order_id"].nunique())
print("Number of unique customers:", delivered_sales["customer_unique_id"].nunique())


Delivered sales shape: (96478, 29)
Number of unique orders: 96478
Number of unique customers: 93358


In [ ]:
total_revenue = delivered_sales["total_order_value"].sum()
average_order_value = delivered_sales["total_order_value"].mean()
median_order_value = delivered_sales["total_order_value"].median()
average_review_score = delivered_sales["review_score_mean"].mean()
average_delivery_delay = delivered_sales["delivery_delay_days"].mean()

print(f"Total revenue: {total_revenue:.2f}")
print(f"Average order value: {average_order_value:.2f}")
print(f"Median order value: {median_order_value:.2f}")
print(f"Average review score: {average_review_score:.2f}")
print(f"Average delivery delay (days): {average_delivery_delay:.2f}")


Total revenue: 15419773.75
Average order value: 159.83
Median order value: 105.28
Average review score: 4.16
Average delivery delay (days): 12.09


In [ ]:
monthly_sales = (
    delivered_sales
    .groupby(["purchase_year", "purchase_month"], as_index=False)
    .agg(
        monthly_revenue=("total_order_value", "sum"),
        monthly_orders=("order_id", "count"),
    )
    .sort_values(["purchase_year", "purchase_month"])
)

monthly_sales.head(12)


,purchase_year,purchase_month,monthly_revenue,monthly_orders
0,2016,9,143.46,1
1,2016,10,46490.66,265
2,2016,12,19.62,1
3,2017,1,127482.37,750
4,2017,2,271239.32,1653
5,2017,3,414330.95,2546
6,2017,4,390812.40,2303
7,2017,5,566851.40,3546
8,2017,6,490050.37,3135
9,2017,7,566299.08,3872


In [ ]:
monthly_sales["year_month"] = pd.to_datetime(
    monthly_sales["purchase_year"].astype(str) + "-" +
    monthly_sales["purchase_month"].astype(str) + "-01"
)

monthly_sales[["year_month", "monthly_revenue", "monthly_orders"]].head()


,year_month,monthly_revenue,monthly_orders
0,2016-09-01,143.46,1
1,2016-10-01,46490.66,265
2,2016-12-01,19.62,1
3,2017-01-01,127482.37,750
4,2017-02-01,271239.32,1653


In [ ]:
state_sales = (
    delivered_sales
    .groupby("customer_state", as_index=False)
    .agg(
        total_revenue=("total_order_value", "sum"),
        total_orders=("order_id", "count"),
        avg_order_value=("total_order_value", "mean"),
    )
    .sort_values("total_revenue", ascending=False)
)

state_sales.head(10)


,customer_state,total_revenue,total_orders,avg_order_value
25,SP,5769703.15,40501,142.458289
18,RJ,2055401.57,12350,166.429277
10,MG,1818891.67,11354,160.198315
22,RS,861472.79,5345,161.173581
17,PR,781708.80,4923,158.787081
23,SC,595127.78,3546,167.830733
4,BA,591137.81,3256,181.553381
6,DF,346123.35,2080,166.405457
8,GO,334212.35,1957,170.777900
7,ES,317657.93,1995,159.227033


In [ ]:
city_sales = (
    delivered_sales
    .groupby("customer_city", as_index=False)
    .agg(
        total_revenue=("total_order_value", "sum"),
        total_orders=("order_id", "count"),
    )
    .sort_values("total_revenue", ascending=False)
)

city_sales.head(10)


,customer_city,total_revenue,total_orders
3563,sao paulo,2107960.17,15045
3126,rio de janeiro,1111732.21,6601
449,belo horizonte,405950.51,2697
553,brasilia,345199.05,2071
1135,curitiba,238459.72,1489
2936,porto alegre,214805.84,1342
700,campinas,209002.90,1406
3218,salvador,207713.30,1188
1518,guarulhos,157735.65,1144
2440,niteroi,135447.96,825


In [ ]:
logistics_kpis = delivered_sales[[
    "delivery_delay_days",
    "estimated_vs_actual_diff_days"
]].describe()

logistics_kpis


,delivery_delay_days,estimated_vs_actual_diff_days
count,96470.000000,96470.000000
mean,12.093604,-11.875889
std,9.551380,10.182105
min,0.000000,-147.000000
25%,6.000000,-17.000000
50%,10.000000,-12.000000
75%,15.000000,-7.000000
max,209.000000,188.000000


In [ ]:
late_delivery_rate = (
    (delivered_sales["estimated_vs_actual_diff_days"] > 0).mean() * 100
)

print(f"Late delivery rate: {late_delivery_rate:.2f}%")


Late delivery rate: 6.77%


In [ ]:
delivered_sales["is_late"] = delivered_sales["estimated_vs_actual_diff_days"] > 0

review_by_late_status = (
    delivered_sales
    .groupby("is_late", as_index=False)
    .agg(
        avg_review_score=("review_score_mean", "mean"),
        avg_delivery_delay=("delivery_delay_days", "mean"),
        n_orders=("order_id", "count"),
    )
)

review_by_late_status


,is_late,avg_review_score,avg_delivery_delay,n_orders
0,False,4.290608,10.539261,89944
1,True,2.271823,33.488062,6534


In [ ]:
monthly_sales["avg_order_value"] = (
    monthly_sales["monthly_revenue"] / monthly_sales["monthly_orders"]
)

monthly_sales[["year_month", "monthly_revenue", "monthly_orders", "avg_order_value"]].head(12)


,year_month,monthly_revenue,monthly_orders,avg_order_value
0,2016-09-01,143.46,1,143.460000
1,2016-10-01,46490.66,265,175.436453
2,2016-12-01,19.62,1,19.620000
3,2017-01-01,127482.37,750,169.976493
4,2017-02-01,271239.32,1653,164.089123
5,2017-03-01,414330.95,2546,162.738001
6,2017-04-01,390812.40,2303,169.697091
7,2017-05-01,566851.40,3546,159.856571
8,2017-06-01,490050.37,3135,156.315907
9,2017-07-01,566299.08,3872,146.254928


## Etape 04 - Analyse commerciale et indicateurs clés

Cette étape a permis de produire les premiers indicateurs business à partir de la table `delivered_sales`, construite au niveau commande et limitée aux ventes effectivement livrées. Les KPI globaux montrent un chiffre d’affaires total de plus de 15,4 millions, un panier moyen d’environ 159,83 et une note moyenne client de 4,16 sur 5.

L’analyse temporelle met en évidence une montée progressive de l’activité à partir de 2017, tandis que l’analyse géographique montre une forte concentration des ventes dans les grands états et centres urbains, notamment Sao Paulo, Rio de Janeiro et Minas Gerais.

L’un des résultats les plus importants concerne la logistique : les commandes sont en moyenne livrées avant la date estimée, et le taux de retard reste relativement faible. En revanche, les commandes livrées en retard présentent une forte baisse de satisfaction client. Cette observation suggère que la performance logistique joue un rôle majeur dans l’expérience client et constitue un levier d’amélioration stratégique pour un acteur e-commerce.

Cette étape confirme que la table analytique construite précédemment est exploitable pour produire des analyses métier pertinentes et prépare directement les prochaines phases du projet, notamment le travail en SQL, la visualisation et la modélisation prédictive.


## 05 - SQL Database Integration and Business Queries


In [ ]:
import sqlite3
from pathlib import Path

DB_PATH = Path("../data/processed/ecommerce.db")
conn = sqlite3.connect(DB_PATH)

print(DB_PATH)


..\data\processed\ecommerce.db


In [ ]:
delivered_sales.to_sql("delivered_sales", conn, if_exists="replace", index=False)

print("Table delivered_sales loaded into SQLite.")


Table delivered_sales loaded into SQLite.


In [ ]:
query = """
SELECT name
FROM sqlite_master
WHERE type='table';
"""

pd.read_sql_query(query, conn)


,name
0,delivered_sales


In [ ]:
query = """
SELECT COUNT(*) AS n_orders
FROM delivered_sales;
"""

pd.read_sql_query(query, conn)


,n_orders
0,96478


In [ ]:
query = """
SELECT
    COUNT(*) AS total_orders,
    COUNT(DISTINCT customer_unique_id) AS unique_customers,
    ROUND(SUM(total_order_value), 2) AS total_revenue,
    ROUND(AVG(total_order_value), 2) AS avg_order_value,
    ROUND(AVG(review_score_mean), 2) AS avg_review_score,
    ROUND(AVG(delivery_delay_days), 2) AS avg_delivery_delay_days
FROM delivered_sales;
"""

pd.read_sql_query(query, conn)


,total_orders,unique_customers,total_revenue,avg_order_value,avg_review_score,avg_delivery_delay_days
0,96478,93358,15419773.75,159.83,4.16,12.09


In [ ]:
query = """
SELECT
    purchase_year,
    purchase_month,
    ROUND(SUM(total_order_value), 2) AS monthly_revenue,
    COUNT(order_id) AS monthly_orders
FROM delivered_sales
GROUP BY purchase_year, purchase_month
ORDER BY purchase_year, purchase_month;
"""

monthly_sales_sql = pd.read_sql_query(query, conn)
monthly_sales_sql.head(12)


,purchase_year,purchase_month,monthly_revenue,monthly_orders
0,2016,9,143.46,1
1,2016,10,46490.66,265
2,2016,12,19.62,1
3,2017,1,127482.37,750
4,2017,2,271239.32,1653
5,2017,3,414330.95,2546
6,2017,4,390812.40,2303
7,2017,5,566851.40,3546
8,2017,6,490050.37,3135
9,2017,7,566299.08,3872


In [ ]:
query = """
SELECT
    customer_state,
    ROUND(SUM(total_order_value), 2) AS total_revenue,
    COUNT(order_id) AS total_orders,
    ROUND(AVG(total_order_value), 2) AS avg_order_value
FROM delivered_sales
GROUP BY customer_state
ORDER BY total_revenue DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)


,customer_state,total_revenue,total_orders,avg_order_value
0,SP,5769703.15,40501,142.46
1,RJ,2055401.57,12350,166.43
2,MG,1818891.67,11354,160.20
3,RS,861472.79,5345,161.17
4,PR,781708.80,4923,158.79
5,SC,595127.78,3546,167.83
6,BA,591137.81,3256,181.55
7,DF,346123.35,2080,166.41
8,GO,334212.35,1957,170.78
9,ES,317657.93,1995,159.23


In [ ]:
query = """
SELECT
    customer_city,
    ROUND(SUM(total_order_value), 2) AS total_revenue,
    COUNT(order_id) AS total_orders
FROM delivered_sales
GROUP BY customer_city
ORDER BY total_revenue DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)


,customer_city,total_revenue,total_orders
0,sao paulo,2107960.17,15045
1,rio de janeiro,1111732.21,6601
2,belo horizonte,405950.51,2697
3,brasilia,345199.05,2071
4,curitiba,238459.72,1489
5,porto alegre,214805.84,1342
6,campinas,209002.90,1406
7,salvador,207713.30,1188
8,guarulhos,157735.65,1144
9,niteroi,135447.96,825


In [ ]:
query = """
SELECT
    ROUND(
        100.0 * SUM(CASE WHEN estimated_vs_actual_diff_days > 0 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS late_delivery_rate_pct
FROM delivered_sales;
"""

pd.read_sql_query(query, conn)


,late_delivery_rate_pct
0,6.77


In [ ]:
query = """
SELECT
    CASE
        WHEN estimated_vs_actual_diff_days > 0 THEN 'late'
        ELSE 'on_time_or_early'
    END AS delivery_status,
    ROUND(AVG(review_score_mean), 2) AS avg_review_score,
    ROUND(AVG(delivery_delay_days), 2) AS avg_delivery_delay,
    COUNT(*) AS n_orders
FROM delivered_sales
GROUP BY delivery_status;
"""

pd.read_sql_query(query, conn)


,delivery_status,avg_review_score,avg_delivery_delay,n_orders
0,late,2.27,33.49,6534
1,on_time_or_early,4.29,10.54,89944


In [ ]:
conn.close()
print("SQLite connection closed.")


SQLite connection closed.


## Etape 05 - Intégration de bases de données SQL et requêtes métier

Cette étape a permis d’intégrer la table analytique `delivered_sales` dans une base SQLite afin de structurer davantage le projet et de reproduire les analyses métier en SQL. La création de la base et le chargement des données se sont déroulés correctement, et les requêtes SQL ont permis de retrouver les mêmes indicateurs que ceux calculés en Python.

Les analyses SQL confirment les principaux résultats observés précédemment : un chiffre d’affaires supérieur à 15,4 millions, une concentration des ventes dans les états et villes les plus importants, une montée de l’activité à partir de 2017, ainsi qu’un taux de retard relativement faible.

Le résultat le plus marquant reste le lien entre la performance logistique et la satisfaction client. Les commandes en retard présentent une note moyenne nettement plus faible que les commandes livrées à l’heure ou en avance. Cette étape montre que les données préparées peuvent désormais être exploitées dans un environnement SQL, ce qui prépare efficacement la suite du projet, notamment le dashboarding et la modélisation.
